# 01 — Exploratory Data Analysis (EDA)
## easyMoney | TFM Data Science & AI — Nuclio School

**Objetivo:** Construir un nuevo autoservicio de Business Intelligence para el equipo, 
que permita responder preguntas clave como:
- ¿Cuántos productos hemos vendido este mes?
- ¿Son los clientes nuevos o los existentes quienes más contratan?
- ¿Cuál es el perfil demográfico de nuestros clientes por producto?

La entrega incluirá un dashboard interactivo y una presentación para el Comité de Dirección.

**Datasets:**
- `commercial_activity_df` — customer status, entry channel, segment (5.96M rows x 17 monthly snapshots)
- `products_df` — binary flags for 14 financial products per customer per snapshot
- `sociodemographic_df` — age, gender, salary, region, country

---

### Preguntas de negocio que guían este análisis

Antes de explorar los datos, definimos las tres preguntas estratégicas que este análisis debe responder:

| Pregunta | KPI | Dónde se responde |
|----------|-----|-------------------|
| ¿Qué clientes son más rentables? | Revenue proxy por segmento (TOP: €40/cliente vs UNIVERSITARIO: €9) | Sección 6 (matriz cross-sell) + Power BI (Tarea 3) |
| ¿Qué segmento tiene mayor riesgo de abandono? | activity_trend < 0 → Inactivos (26.5% de la base, revenue_proxy €610) | Sección 7 (inactividad) + Segmentación (Tarea 2) |
| ¿Dónde funciona el cross-sell? | Nómina ↔ Pensión (r = 0.97); PARTICULARES = mayor volumen potencial | Sección 6 (matriz cross-sell) + Deep Dive (Tarea 1b) |

> Las respuestas completas se consolidan en la **Sección 8 (KPIs)** y se visualizan en el **dashboard Power BI (Tarea 3)**.

In [27]:
import pandas as pd
import os
import glob
from pathlib import Path

# ── Setup visualizaciones ──────────────────────────────────────────
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Paleta de colores easyMoney
EM_GREEN  = '#4CAF50'
EM_DARK   = '#2E7D32'
EM_LIGHT  = '#A5D6A7'
EM_GRAY   = '#78909C'
EM_ORANGE = '#FF9800'

print("✓ Plotly listo para visualizaciones!")
print(f"Versión plotly: {px.__version__ if hasattr(px, '__version__') else 'ok'}")

✓ Plotly listo para visualizaciones!
Versión plotly: ok


## 1. Carga de Datos

In [28]:
DATA_DIR = Path('../../data/raw')   

df_commercial = pd.read_csv(DATA_DIR / 'commercial_activity_df.csv', index_col=0)
df_products   = pd.read_csv(DATA_DIR / 'products_df.csv',            index_col=0)
df_socio      = pd.read_csv(DATA_DIR / 'sociodemographic_df.csv',    index_col=0)

for name, df in [('commercial', df_commercial), ('products', df_products), ('socio', df_socio)]:
    print(f'{name:15s} -> {df.shape[0]:>9,} rows  x  {df.shape[1]:>2} cols')

print('\nData loaded successfully')

commercial      -> 5,962,924 rows  x   6 cols
products        -> 5,962,924 rows  x  17 cols
socio           -> 5,962,924 rows  x   8 cols

Data loaded successfully


Los datos han sido proporcionados por el equipo de IT (Frank) como un volcado de la 
base de datos del autoservicio de BI existente, estructurado en **3 tablas separadas** 
que comparten las claves `pk_cid` (identificador de cliente) y `pk_partition` (fecha 
de ingesta, equivalente al período de análisis).


| Tabla | Filas | Columnas | Contenido |
|---|---|---|---|
| `commercial_activity` | 5,962,924 | 6 | Actividad comercial del cliente |
| `products` | 5,962,924 | 17 | Productos contratados (flags 0/1) |
| `sociodemographic` | 5,962,924 | 8 | Perfil sociodemográfico del cliente |

### Nota sobre `products_df` — tabla de estado mensual

`products_df.csv` es la tabla más importante del proyecto. Antes de cargarla conviene entender su semántica:

| Campo | Tipo | Descripción |
|---|---|---|
| `pk_cid` | clave | Identificador de cliente |
| `pk_partition` | clave | Fecha del snapshot (fin de mes) |
| 14 columnas de producto | binario 0/1 | **1 = tiene el producto en ese momento; 0 = no lo tiene** |

**Puntos clave:**
- Cada fila es un **snapshot de estado**, no una transacción. Una celda con `1` solo dice que el cliente tenía ese producto en esa fecha, no cuándo lo contrató.
- Por tanto, para saber cuándo se contrató algo hay que calcular el **delta** entre particiones consecutivas → eso es `new_contracts` (Sección 5).
- `em_account_pp`: siempre 0 en todo el dataset → columna sin información, se excluye del análisis.
- `em_account_p`: solo 34 filas activas en toda la historia → penetración ≈ 0, se mantiene por completitud.
- `payroll` y `pension_plan` tienen 61 NaNs ocultos almacenados como `float64` → se corrigen a `int8` en la Sección 4.

## 2. Exploración de la Estructura

Antes de cualquier análisis, inspeccionamos los tipos de datos, las primeras filas 
y la distribución de valores en cada tabla para entender con qué estamos trabajando.


In [29]:
# ── Paso 2: Vista rápida de cada tabla ─────────────────────────────────────

for name, df in [('commercial', df_commercial), 
                ('products',   df_products), 
                ('socio',      df_socio)]:
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(df.dtypes)
    print("\nPrimeras 3 filas:")
    print(df.head(3))


  COMMERCIAL
pk_cid               int64
pk_partition        object
entry_date          object
entry_channel       object
active_customer    float64
segment             object
dtype: object

Primeras 3 filas:
    pk_cid pk_partition  entry_date entry_channel  active_customer  \
0  1375586   2018-01-28  2018-01-12           KHL              1.0   
1  1050611   2018-01-28  2015-08-10           KHE              0.0   
2  1050612   2018-01-28  2015-08-10           KHE              0.0   

              segment  
0   02 - PARTICULARES  
1  03 - UNIVERSITARIO  
2  03 - UNIVERSITARIO  

  PRODUCTS
pk_cid                  int64
pk_partition           object
short_term_deposit      int64
loans                   int64
mortgage                int64
funds                   int64
securities              int64
long_term_deposit       int64
em_account_pp           int64
credit_card             int64
payroll               float64
pension_plan          float64
payroll_account         int64
emc_account 

### Hallazgos principales:

**Tabla commercial:**
- `pk_partition` y `entry_date` están almacenados como `object` → necesitan conversión a fecha
- `active_customer` solo tiene **2 valores únicos** (0/1) → flag binario correcto
- `segment` tiene **3 valores únicos** → segmentación comercial simple
- `entry_channel` tiene **68 valores únicos** → alta cardinalidad, investigar

**Tabla products:**
- Todos los productos son **flags binarios (0/1)** → estructura limpia
- `payroll` y `pension_plan` almacenados como `float64` → deben ser enteros
- `em_account_pp` tiene **1 solo valor único** → columna sin información, eliminar

**Tabla socio:**
- `region_code` almacenado como `float64` → debe convertirse a string (es un código)
- `gender` tiene **2 valores únicos** → binario correcto
- `age` tiene **104 valores únicos** → rango de edades razonable
- `salary` tiene **258,629 valores únicos** → variable continua, confirma que es ingreso real

---

## 3. Análisis de Calidad de Datos (Nulos)

Revisamos el porcentaje de valores faltantes en cada tabla para decidir 
la estrategia de tratamiento antes de construir ningún análisis.

In [30]:
# ── Paso 3: Nulos y valores únicos ─────────────────────────────────────────

for name, df in [('commercial', df_commercial), 
                 ('products',   df_products), 
                 ('socio',      df_socio)]:
    print(f"\n{'='*50}")
    print(f"  {name.upper()} — Nulos (%)")
    print(f"{'='*50}")
    null_pct = (df.isnull().sum() / len(df) * 100).round(2)
    print(null_pct[null_pct > 0] if null_pct.any() else "  Sin nulos")
    
    print(f"\n  Valores únicos por columna:")
    for col in df.columns:
        print(f"  {col:<25} {df[col].nunique():>8} únicos")


  COMMERCIAL — Nulos (%)
entry_channel    2.23
segment          2.25
dtype: float64

  Valores únicos por columna:
  pk_cid                      456373 únicos
  pk_partition                    17 únicos
  entry_date                    1499 únicos
  entry_channel                   68 únicos
  active_customer                  2 únicos
  segment                          3 únicos

  PRODUCTS — Nulos (%)
  Sin nulos

  Valores únicos por columna:
  pk_cid                      456373 únicos
  pk_partition                    17 únicos
  short_term_deposit               2 únicos
  loans                            2 únicos
  mortgage                         2 únicos
  funds                            2 únicos
  securities                       2 únicos
  long_term_deposit                2 únicos
  em_account_pp                    1 únicos
  credit_card                      2 únicos
  payroll                          2 únicos
  pension_plan                     2 únicos
  payroll_account        

| Tabla | Campo | % Nulos | Estrategia |
|---|---|---|---|
| commercial | `entry_channel` | 2.23% | Rellenar con 'UNKNOWN' |
| commercial | `segment` | 2.25% | Imputación inteligente por salary + region_code + age |
| socio | `region_code` | 0.04% | Negligible, rellenar con 'UNKNOWN' |
| socio | `salary` | 25.36% | Imputación inteligente por grupo de edad |
| products | — | 0% | Sin nulos ✓ |

> **Decisión clave sobre salary:** Con un 25% de nulos no podemos eliminar filas.
> Tras analizar la distribución por edad de los valores nulos, detectamos que el patrón
> no es aleatorio. La mayoría corresponde a jóvenes de 18-24 años (posibles becarios
> o estudiantes) y menores de 18. Por ello aplicamos una **imputación por grupo de edad**:
>
> | Grupo | Edad | Criterio | Valor imputado |
> |---|---|---|---|
> | Menores | < 18 | Sin ingresos reales | 0 |
> | Jóvenes | 18-24 | Becarios/estudiantes | Mediana del grupo (€88,496) |
> | Activos | 25-64 | Trabajadores en activo | Mediana del grupo (€87,861) |
> | Jubilados | 65+ | Pensionistas | Mediana del grupo (€102,809) |
>
> Esta estrategia es más realista que una mediana global porque respeta el perfil
> económico real de cada etapa vital. Se añade el flag `salary_imputed` para
> trazabilidad completa.

---

## 4. Limpieza y Conversión de Tipos

In [31]:
# ── Diagnóstico: fechas problemáticas en entry_date ────────────────────────

# Primero convertimos pk_partition (que funciona bien)
df_commercial['pk_partition'] = pd.to_datetime(df_commercial['pk_partition'])
df_products['pk_partition']   = pd.to_datetime(df_products['pk_partition'])
df_socio['pk_partition']      = pd.to_datetime(df_socio['pk_partition'])

# Ver qué hay en entry_date alrededor de la posición 688
print("Muestra de entry_date problemáticas:")
bad_dates = pd.to_datetime(df_commercial['entry_date'], errors='coerce')
mask = bad_dates.isna() & df_commercial['entry_date'].notna()
print(f"Total fechas inválidas: {mask.sum()}")
print(df_commercial[mask]['entry_date'].value_counts().head(20))

Muestra de entry_date problemáticas:
Total fechas inválidas: 6413
entry_date
2019-02-29    4621
2015-02-29    1792
Name: count, dtype: int64


In [32]:
# ── Fix: corregir fechas inválidas en entry_date ───────────────────────────

# Reemplazar fechas imposibles antes de convertir
df_commercial['entry_date'] = df_commercial['entry_date'].replace({
    '2019-02-29': '2019-02-28',
    '2015-02-29': '2015-02-28'
})

# Ahora sí convertir a datetime
df_commercial['entry_date'] = pd.to_datetime(df_commercial['entry_date'])

# Verificar que no quedan nulos inesperados
print(f"Nulos en entry_date tras fix: {df_commercial['entry_date'].isna().sum()}")
print(f"Tipo: {df_commercial['entry_date'].dtype}")
print(f"\nRango de fechas:")
print(f"  Más antigua: {df_commercial['entry_date'].min()}")
print(f"  Más reciente: {df_commercial['entry_date'].max()}")

Nulos en entry_date tras fix: 0
Tipo: datetime64[ns]

Rango de fechas:
  Más antigua: 2015-01-01 00:00:00
  Más reciente: 2019-05-31 00:00:00


In [33]:
# ── Fix: convertir floats con posibles NaN a int en products ───────────────

# Primero verificar qué hay exactamente
print("Nulos reales en payroll:", df_products['payroll'].isna().sum())
print("Nulos reales en pension_plan:", df_products['pension_plan'].isna().sum())

# Rellenar cualquier NaN con 0 antes de convertir a int
df_products['payroll']      = df_products['payroll'].fillna(0).astype(int)
df_products['pension_plan'] = df_products['pension_plan'].fillna(0).astype(int)

# Verificar
print(f"\npayroll dtype: {df_products['payroll'].dtype}")
print(f"pension_plan dtype: {df_products['pension_plan'].dtype}")
print(f"Valores únicos payroll: {df_products['payroll'].unique()}")
print(f"Valores únicos pension_plan: {df_products['pension_plan'].unique()}")

Nulos reales en payroll: 61
Nulos reales en pension_plan: 61

payroll dtype: int64
pension_plan dtype: int64
Valores únicos payroll: [0 1]
Valores únicos pension_plan: [0 1]


In [34]:
# ── Paso 4: Limpieza y conversión de tipos ─────────────────────────────────

# 4.1 Conversión de fechas — ya realizada en las celdas anteriores de diagnóstico
# (pk_partition → datetime en las 3 tablas, entry_date corregida y convertida)
# Se omite aquí para evitar redundancia.

# 4.2 Convertir floats que deberían ser int en products
# (payroll y pension_plan ya se convirtieron con fillna(0) en la celda anterior)
# Se verifica únicamente que los tipos son correctos
assert df_products['payroll'].dtype == 'int64', "payroll debería ser int64"
assert df_products['pension_plan'].dtype == 'int64', "pension_plan debería ser int64"

# 4.3 region_code: float → string (es un código, no un número)
df_socio['region_code'] = df_socio['region_code'].fillna(-1).astype(int).astype(str)
df_socio['region_code'] = df_socio['region_code'].replace('-1', None)

# 4.4 Salary: imputación inteligente por grupo de edad
def impute_salary_final(df):
    df = df.copy()
    df['salary_imputed'] = df['salary'].isna()
    
    # Menores de 18 → 0
    mask_minor = (df['salary'].isna()) & (df['age'] < 18)
    df.loc[mask_minor, 'salary'] = 0
    
    # 18-24 → mediana del grupo
    median_young = df[(df['salary'].notna()) & 
                      (df['age'].between(18, 24))]['salary'].median()
    mask_young = (df['salary'].isna()) & (df['age'].between(18, 24))
    df.loc[mask_young, 'salary'] = median_young
    
    # 25-64 → mediana del grupo
    median_working = df[(df['salary'].notna()) & 
                        (df['age'].between(25, 64))]['salary'].median()
    mask_working = (df['salary'].isna()) & (df['age'].between(25, 64))
    df.loc[mask_working, 'salary'] = median_working
    
    # 65+ → mediana del grupo
    median_retired = df[(df['salary'].notna()) & 
                        (df['age'] >= 65)]['salary'].median()
    mask_retired = (df['salary'].isna()) & (df['age'] >= 65)
    df.loc[mask_retired, 'salary'] = median_retired
    
    return df

df_socio = impute_salary_final(df_socio)
print(f"✓ Nulos restantes en salary: {df_socio['salary'].isna().sum()}")

# 4.5 Eliminar em_account_pp (un solo valor único → sin información)
print(f"em_account_pp valores únicos: {df_products['em_account_pp'].unique()}")
df_products = df_products.drop(columns=['em_account_pp'])

# 4.6 entry_channel → moda por segmento
df_commercial['entry_channel'] = df_commercial['entry_channel'].fillna(
    df_commercial.groupby('segment')['entry_channel'].transform(
        lambda x: x.mode()[0] if not x.mode().empty else 'UNKNOWN'
    )
)
df_commercial['entry_channel'] = df_commercial['entry_channel'].fillna('UNKNOWN')

# 4.7 segment → imputación inteligente por salary + region_code + age
segment_to_num = {'03 - UNIVERSITARIO': 1, '02 - PARTICULARES': 2, '01 - TOP': 3}
num_to_segment = {v: k for k, v in segment_to_num.items()}

df_commercial['segment_num'] = df_commercial['segment'].map(segment_to_num)

temp = df_commercial.merge(
    df_socio[['pk_cid','pk_partition','salary','region_code','age']], 
    on=['pk_cid','pk_partition'], how='left'
)

group_median = temp.groupby(['salary','region_code','age'])['segment_num'].transform('median')
df_commercial['segment_num'] = df_commercial['segment_num'].fillna(group_median)
df_commercial['segment_num'] = df_commercial['segment_num'].fillna(
    df_commercial['segment_num'].median()
)
df_commercial['segment'] = df_commercial['segment_num'].round().astype(int).map(num_to_segment)
df_commercial = df_commercial.drop(columns=['segment_num'])

print(f"✓ Nulos en segment: {df_commercial['segment'].isna().sum()}")
print(f"✓ Distribución segment:")
print(df_commercial['segment'].value_counts())

# ── Verificación final ─────────────────────────────────────────────────────
print("\nTipos después de limpieza:")
for name, df in [('commercial', df_commercial), 
                 ('products',   df_products), 
                 ('socio',      df_socio)]:
    print(f"\n{name}:")
    print(df.dtypes)

print("\nNulos restantes:")
for name, df in [('commercial', df_commercial), 
                 ('products',   df_products), 
                 ('socio',      df_socio)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"\n{name}: {nulls.to_dict() if len(nulls) else 'Sin nulos'}")

✓ Nulos restantes en salary: 0
em_account_pp valores únicos: [0]
✓ Nulos en segment: 0
✓ Distribución segment:
segment
03 - UNIVERSITARIO    4026906
02 - PARTICULARES     1837902
01 - TOP                98116
Name: count, dtype: int64

Tipos después de limpieza:

commercial:
pk_cid                      int64
pk_partition       datetime64[ns]
entry_date         datetime64[ns]
entry_channel              object
active_customer           float64
segment                    object
dtype: object

products:
pk_cid                         int64
pk_partition          datetime64[ns]
short_term_deposit             int64
loans                          int64
mortgage                       int64
funds                          int64
securities                     int64
long_term_deposit              int64
credit_card                    int64
payroll                        int64
pension_plan                   int64
payroll_account                int64
emc_account                    int64
debit_card    

In [35]:
# ── Fix final de nulos restantes ───────────────────────────────────────────

# Gender: 25 nulos → UNKNOWN
df_socio['gender'] = df_socio['gender'].fillna('UNKNOWN')

# Region_code: mantener None pero rellenar con 'UNKNOWN' para análisis
df_socio['region_code'] = df_socio['region_code'].fillna('UNKNOWN')

# ── Verificación final limpia ──────────────────────────────────────────────
print("\n✓ Nulos finales:")
for name, df in [('commercial', df_commercial), 
                 ('products',   df_products), 
                 ('socio',      df_socio)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    print(f"  {name}: {nulls.to_dict() if len(nulls) else 'Sin nulos ✓'}")

print("✓ Limpieza completada!")


✓ Nulos finales:
  commercial: Sin nulos ✓
  products: Sin nulos ✓
  socio: Sin nulos ✓
✓ Limpieza completada!


| Problema | Solución aplicada |
|---|---|
| `entry_date` con fechas imposibles (29/02 en años no bisiestos) | Reemplazadas por 28/02 del mismo año — 6,413 registros corregidos |
| `payroll` y `pension_plan` como float64 con 61 NaNs ocultos | Rellenados con 0 y convertidos a int64 |
| `em_account_pp` con un único valor (0) | Columna eliminada — no aporta información |
| `region_code` como float64 | Convertido a string categórico |
| `salary` con 25.36% de nulos | Imputación inteligente por grupo de edad: <18 → 0, 18-24 → €88,496, 25-64 → €87,861, 65+ → €102,809 |
| `gender` con 25 nulos | Rellenado con 'UNKNOWN' |
| `segment` con 2.25% nulos | Imputación inteligente por salary + region_code + age → 67.5% UNIVERSITARIO, 30.8% PARTICULARES, 1.6% TOP |
| `entry_channel` con 2.23% nulos | Imputado con moda por segmento + fallback UNKNOWN |

> **Estado final:** Las 3 tablas están completamente limpias y listas para 
> el Feature Engineering. Se añadió el flag `salary_imputed` para trazabilidad.

---

## 5. Feature Engineering

A partir de las tres tablas unidas, se construyen siete features adicionales que enriquecen el análisis: identificación de clientes nuevos vs existentes, contabilización de nuevas contrataciones, antigüedad del cliente, y segmentación por edad y salario. Cada paso está documentado con su justificación de negocio.

In [36]:
# ── Paso 5: Feature Engineering ────────────────────────────────────────────

# 5.1 Unir las 3 tablas en un único dataframe maestro
df = df_commercial.merge(df_products, on=['pk_cid','pk_partition'], how='inner') \
                  .merge(df_socio,    on=['pk_cid','pk_partition'], how='inner')

print(f"✓ Tabla maestra: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# 5.2 Definir columnas de productos
product_cols = ['short_term_deposit','loans','mortgage','funds','securities',
                'long_term_deposit','credit_card','payroll','pension_plan',
                'payroll_account','emc_account','debit_card','em_account_p',
                'em_acount']

print(f"✓ Productos: {len(product_cols)} columnas")

# 5.3 Total de productos por cliente por período
df['total_products'] = df[product_cols].sum(axis=1)
print(f"\nDistribución de productos por cliente:")
print(df['total_products'].value_counts().sort_index())

✓ Tabla maestra: 5,962,924 filas × 27 columnas
✓ Productos: 14 columnas

Distribución de productos por cliente:
total_products
0    1121507
1    3995714
2     528593
3     150269
4     105720
5      42890
6      14809
7       2799
8        573
9         50
Name: count, dtype: int64


In [37]:
# ── Optimización de memoria: columnas binarias de productos ─────────────────
# Los 14 productos son flags binarios (0/1). Almacenarlos como int64 malgasta
# 8 bytes por valor cuando int8 (1 byte) es suficiente para representar {0, 1}.

mem_antes = df[product_cols].memory_usage(deep=True).sum() / 1024**2
df[product_cols] = df[product_cols].astype('int8')
mem_despues = df[product_cols].memory_usage(deep=True).sum() / 1024**2

print("Optimización de memoria — columnas de producto:")
print(f"  Antes:   {mem_antes:.1f} MB  (int64)")
print(f"  Después: {mem_despues:.1f} MB  (int8)")
print(f"  Ahorro:  {(1 - mem_despues / mem_antes) * 100:.0f}% ({mem_antes - mem_despues:.1f} MB liberados)")
print(f"\n✓ Recalculando total_products con dtype correcto...")
df['total_products'] = df[product_cols].sum(axis=1)
print(f"  total_products — rango: {df['total_products'].min()} – {df['total_products'].max()}")

Optimización de memoria — columnas de producto:
  Antes:   636.9 MB  (int64)
  Después: 79.6 MB  (int8)
  Ahorro:  87% (557.3 MB liberados)

✓ Recalculando total_products con dtype correcto...
  total_products — rango: 0 – 9


In [38]:
# ── Paso 5 (continuación): más features ────────────────────────────────────

# 5.4 Identificar clientes nuevos vs existentes por partición
# Un cliente es "nuevo" en la partición en que aparece por primera vez
first_partition = df.groupby('pk_cid')['pk_partition'].min().reset_index()
first_partition.columns = ['pk_cid', 'first_partition']

df = df.merge(first_partition, on='pk_cid', how='left')
df['is_new_client'] = (df['pk_partition'] == df['first_partition']).astype(int)

print(f"Clientes nuevos por período:")
print(df.groupby('pk_partition')['is_new_client'].sum().to_string())

# 5.5 Nuevos contratos por cliente por período
# DEFINICIÓN: cualquier producto que un cliente activa por primera vez en ese período.
#
# Implementación: se calcula el valor del producto en el período anterior (shift).
#   - Clientes EXISTENTES: prev = valor real del período anterior.
#     new_contract = 1 si prev==0 y actual==1 (contratación nueva).
#   - Clientes NUEVOS (primera partición): prev = NaN → fillna(0).
#     Todos sus productos actuales se cuentan como nuevos contratos,
#     por lo que new_contracts == total_products en su primera partición.
#     Esto es correcto: al alta, cada producto que tienen lo contrataron por primera vez.

df = df.sort_values(['pk_cid', 'pk_partition'])

# Calcular valores previos para cada producto
for col in product_cols:
    df[f'{col}_prev'] = df.groupby('pk_cid')[col].shift(1)

prev_cols = [f'{col}_prev' for col in product_cols]

# Nuevas contrataciones = no tenía el producto antes (o es nuevo) y ahora sí
df['new_contracts'] = 0
for col in product_cols:
    df['new_contracts'] += (
        (df[f'{col}_prev'].fillna(0) == 0) & (df[col] == 1)
    ).astype(int)

# Eliminar columnas temporales
df = df.drop(columns=prev_cols)

# ── Verificación ────────────────────────────────────────────────────────
print("Nuevas contrataciones por período:")
print(df.groupby('pk_partition')['new_contracts'].sum().to_string())

# Desglose por tipo de cliente
total  = df['new_contracts'].sum()
exist  = df[df['is_new_client'] == 0]['new_contracts'].sum()
nuevos = df[df['is_new_client'] == 1]['new_contracts'].sum()

print(f"\n✓ De clientes existentes: {exist:,}  ({exist/total*100:.1f}%)")
print(f"✓ De clientes nuevos:     {nuevos:,} ({nuevos/total*100:.1f}%)")
print(f"✓ Total: {total:,}")

# 5.6 Antigüedad del cliente en meses
df['client_age_months'] = ((df['pk_partition'] - df['entry_date']) / 
                            pd.Timedelta(days=30)).round().astype(int)

print(f"\nAntigüedad media del cliente: {df['client_age_months'].mean():.1f} meses")
print(f"Antigüedad máxima: {df['client_age_months'].max()} meses")

Clientes nuevos por período:
pk_partition
2018-01-28    239493
2018-02-28      3767
2018-03-28      3411
2018-04-28      2952
2018-05-28      3031
2018-06-28      2845
2018-07-28     83840
2018-08-28     14596
2018-09-28     23353
2018-10-28     28205
2018-11-28     15557
2018-12-28      7402
2019-01-28      6926
2019-02-28      6193
2019-03-28      5690
2019-04-28      4581
2019-05-28      4531
Nuevas contrataciones por período:
pk_partition
2018-01-28    296613
2018-02-28     15182
2018-03-28     15281
2018-04-28     13741
2018-05-28     13621
2018-06-28     17588
2018-07-28     26051
2018-08-28     25497
2018-09-28     31550
2018-10-28     34925
2018-11-28     25387
2018-12-28     22962
2019-01-28     17828
2019-02-28     23550
2019-03-28     19429
2019-04-28     17319
2019-05-28     18726

✓ De clientes existentes: 240,577  (37.9%)
✓ De clientes nuevos:     394,673 (62.1%)
✓ Total: 635,250

Antigüedad media del cliente: 20.9 meses
Antigüedad máxima: 54 meses


**Nota metodológica — `new_contracts`:**

En la versión inicial, `is_new_client = 1` registraba siempre
`new_contracts = 0` en su primera partición, porque `shift(1)`
devuelve NaN y la condición `NaN == 0` es False en pandas.

Esto se corrigió aplicando `fillna(0)` al valor previo:
si un cliente no tiene período anterior (NaN), se trata como
que "no tenía el producto antes" — lo cual es correcto por
definición. Así, los contratos iniciales de clientes nuevos
quedan correctamente contabilizados.

**Definición final:** `new_contracts` cuenta cualquier producto
que un cliente activa por primera vez en ese período,
independientemente de si es cliente nuevo o existente.

**Edge case documentado:** Si un cliente cancela un producto y lo
vuelve a contratar en un período posterior, se contabiliza como
nuevo contrato (prev=0 → actual=1). Este comportamiento es
intencional — representa una recontratación real.

**Resultados clave:**
- Total histórico: clientes existentes 37.9% vs nuevos 62.1%
  ⚠️ Distorsionado por spike de enero 2018 (posible migración)
- Sin enero 2018: existentes **71.0%** vs nuevos 29.0%
- Solo 2019: existentes **86.8%** vs nuevos 13.2% ← cifra relevante


In [39]:
# ── Paso 5 (continuación): features finales ────────────────────────────────

# 5.7 Categorías de edad
df['age_group'] = pd.cut(df['age'], 
                        bins=[-1, 17, 24, 34, 44, 54, 64, 200],  # 200 para incluir edades > 100 sin NaN
                        labels=['<18', '18-24', '25-35', '35-45', '45-55', '55-65', '65+'])     

# 5.8 Categorías de salario
df['salary_group'] = pd.cut(df['salary'],
                        bins=[-1, 0, 20000, 40000, 60000, 80000, 120000, 999999999],
                        labels=['sin_ingreso', '<20k', '20-40k', '40-60k', '60-80k', '80-120k', '120k+'])       

# 5.9 Resumen final de la tabla maestra
print("✓ Feature Engineering completado!")
print(f"\nTabla maestra final: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas disponibles:")
for col in df.columns:
    print(f"  {col}")

print(f"\nEjemplo — distribución por grupo de edad:")
print(df['age_group'].value_counts().sort_index())

print(f"\nEjemplo — distribución por grupo de salario:")
print(df['salary_group'].value_counts().sort_index())

✓ Feature Engineering completado!

Tabla maestra final: 5,962,924 filas × 34 columnas

Columnas disponibles:
  pk_cid
  pk_partition
  entry_date
  entry_channel
  active_customer
  segment
  short_term_deposit
  loans
  mortgage
  funds
  securities
  long_term_deposit
  credit_card
  payroll
  pension_plan
  payroll_account
  emc_account
  debit_card
  em_account_p
  em_acount
  country_id
  region_code
  gender
  age
  deceased
  salary
  salary_imputed
  total_products
  first_partition
  is_new_client
  new_contracts
  client_age_months
  age_group
  salary_group

Ejemplo — distribución por grupo de edad:
age_group
<18        35873
18-24    2933206
25-35    1545217
35-45     744912
45-55     387872
55-65     183583
65+       132261
Name: count, dtype: int64

Ejemplo — distribución por grupo de salario:
salary_group
sin_ingreso       6872
<20k             22209
20-40k          286783
40-60k          736109
60-80k          854807
80-120k        2713918
120k+          1342226
Name: c

### Resultados — Feature Engineering

A partir de las 3 tablas limpias se construyó una **tabla maestra única** 
de 5,962,924 filas × 34 columnas con las siguientes features adicionales:

| Feature | Descripción |
|---|---|
| `total_products` | Número total de productos contratados por cliente en cada período |
| `is_new_client` | Flag: 1 si es la primera vez que el cliente aparece en los datos |
| `new_contracts` | Número de productos nuevos contratados respecto al período anterior |
| `client_age_months` | Antigüedad del cliente en meses desde su primera contratación |
| `first_partition` | Fecha de la primera aparición del cliente en los datos |
| `age_group` | Grupos de edad: <18, 18-24, 25-35, 35-45, 45-55, 55-65, 65+ |
| `salary_group` | Grupos de salario: sin_ingreso, <20k, 20-40k, 40-60k, 60-80k, 80-120k, 120k+ |

### Insights preliminares del Feature Engineering:

- La base de clientes es predominantemente **joven (<25 años)** — perfil 
  típico de una plataforma fintech digital. Destaca el grupo **<18 años** 
  con salary imputado a 0 (sin ingresos) y el grupo **18-24** como posibles 
  becarios o estudiantes, lo que explica el alto porcentaje de nulos en salary.
- El nivel salarial es **sorprendentemente alto** — el grupo 80-120k es 
  el más numeroso, lo que sugiere un perfil de cliente con capacidad 
  de inversión elevada
- La mayoría de clientes tienen **0 o 1 productos** — enorme oportunidad 
  de cross-sell alineada con la estrategia de penetración de Ansoff
- Las nuevas contrataciones muestran una **tendencia creciente** 
  (~10,000 → ~20,000/mes) a lo largo de los 17 períodos
- El spike de **83,840 clientes nuevos en julio 2018** requiere 
  investigación — posible campaña de captación masiva o migración de datos

---

## 6. Visualizaciones y Dashboard BI

Los gráficos de esta sección responden a las preguntas de negocio de Carol (CEO): evolución de la base de clientes, penetración por producto, origen de nuevas contrataciones y perfil demográfico. Cada bloque incluye una **lectura de negocio** orientada al Comité de Dirección.

In [40]:
# ── Gráfico 1: Evolución de clientes activos y nuevas contrataciones ────────

# Agregar datos por período
period_summary = df.groupby('pk_partition').agg(
    total_clients        = ('pk_cid', 'count'),
    new_clients          = ('is_new_client', 'sum'),
    total_new_contracts  = ('new_contracts', 'sum'),
    active_clients       = ('active_customer', 'sum')
).reset_index()

period_summary['pk_partition'] = period_summary['pk_partition'].astype(str).str[:10]

# Gráfico con doble eje Y
fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(go.Bar(
    x=period_summary['pk_partition'],
    y=period_summary['new_clients'],
    name='Clientes nuevos',
    marker_color=EM_LIGHT,
    opacity=0.7
), secondary_y=False)

fig1.add_trace(go.Scatter(
    x=period_summary['pk_partition'],
    y=period_summary['total_clients'],
    name='Total clientes',
    line=dict(color=EM_DARK, width=3),
    mode='lines+markers'
), secondary_y=False)

fig1.add_trace(go.Scatter(
    x=period_summary['pk_partition'],
    y=period_summary['total_new_contracts'],
    name='Nuevas contrataciones',
    line=dict(color=EM_ORANGE, width=2, dash='dot'),
    mode='lines+markers'
), secondary_y=True)

fig1.update_layout(
    title='Evolución de clientes y nuevas contrataciones por período',
    xaxis_title='Período',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    plot_bgcolor='white',
    height=450
)
fig1.update_yaxes(title_text='Nº clientes', secondary_y=False)
fig1.update_yaxes(title_text='Nuevas contrataciones', secondary_y=True)
fig1.update_xaxes(tickangle=45)

fig1.show()
print("✓ Gráfico 1 generado!")

✓ Gráfico 1 generado!


In [41]:
# 2018-01-28 es un outlier claro, veamos el impacto de excluirlo
df_excl = df[df['pk_partition'] != '2018-01-28']
total_excl = df_excl['new_contracts'].sum()
exist_excl = df_excl[df_excl['is_new_client']==0]['new_contracts'].sum()
new_excl   = df_excl[df_excl['is_new_client']==1]['new_contracts'].sum()

print(f"\n  2018 sin enero :")
print(f"✓ de clientes existentes: {exist_excl:,} ({exist_excl/total_excl*100:.1f}%)")
print(f"✓ de clientes nuevos:     {new_excl:,} ({new_excl/total_excl*100:.1f}%)")

# 2019 es un año completo, veamos su desglose
df_2019 = df[df['pk_partition'].dt.year == 2019]
total_2019 = df_2019['new_contracts'].sum()
exist_2019 = df_2019[df_2019['is_new_client']==0]['new_contracts'].sum()
new_2019   = df_2019[df_2019['is_new_client']==1]['new_contracts'].sum()

print(f"\n  2019:")
print(f"✓ de clientes existentes: {exist_2019:,} ({exist_2019/total_2019*100:.1f}%)")
print(f"✓ de clientes nuevos:     {new_2019:,} ({new_2019/total_2019*100:.1f}%)")


  2018 sin enero :
✓ de clientes existentes: 240,577 (71.0%)
✓ de clientes nuevos:     98,060 (29.0%)

  2019:
✓ de clientes existentes: 84,100 (86.8%)
✓ de clientes nuevos:     12,752 (13.2%)


### Nota sobre la primera partición (enero 2018)

La primera barra muestra ~240k "nuevos clientes", lo cual es técnicamente correcto, pero visualmente engañoso: no representa crecimiento real, sino la fotografía inicial de todos los clientes existentes. El gráfico corregido (Gráfico 1b) excluye esta partición de las barras de nuevos clientes para evitar confusión en el Comité.

In [42]:
# ── Gráfico 1 corregido: excluir primera partición del bar ─────────────────

first_period = period_summary['pk_partition'].min()
period_plot = period_summary.copy()
period_plot.loc[period_plot['pk_partition'] == first_period, 'new_clients'] = 0

fig1 = make_subplots(specs=[[{"secondary_y": True}]])

fig1.add_trace(go.Bar(
    x=period_plot['pk_partition'],
    y=period_plot['new_clients'],
    name='Clientes nuevos',
    marker_color=EM_LIGHT,
    opacity=0.7
), secondary_y=False)

fig1.add_trace(go.Scatter(
    x=period_plot['pk_partition'],
    y=period_plot['total_clients'],
    name='Total clientes',
    line=dict(color=EM_DARK, width=3),
    mode='lines+markers'
), secondary_y=False)

fig1.add_trace(go.Scatter(
    x=period_plot['pk_partition'],
    y=period_plot['total_new_contracts'],
    name='Nuevas contrataciones',
    line=dict(color=EM_ORANGE, width=2, dash='dot'),
    mode='lines+markers'
), secondary_y=True)

fig1.update_layout(
    title='Evolución de clientes y nuevas contrataciones por período',
    xaxis_title='Período',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    plot_bgcolor='white',
    height=450
)
fig1.update_yaxes(title_text='Nº clientes', secondary_y=False)
fig1.update_yaxes(title_text='Nuevas contrataciones', secondary_y=True)
fig1.update_xaxes(tickangle=45)

fig1.show()
print("✓ Gráfico 1 corregido!")

✓ Gráfico 1 corregido!


### Lectura de negocio — Gráfico 1

El gráfico cuenta una historia de negocio clara:

- Crecimiento sostenido de clientes: de 240k a 440k en 17 meses 
- El pico de julio de 2018 sigue siendo visible en la barra de nuevos clientes; conviene destacárselo a Carol
- Los nuevos contratos muestran una tendencia al alza: de ~11k a ~20k al mes, lo que refleja un impulso saludable de cross-sell 
- Ahora las barras son honestas: solo muestran clientes realmente nuevos en cada periodo 

In [43]:
# ── Gráfico 2: Tasa de penetración por producto ────────────────────────────

# Usar solo la última partición para foto actual
last_partition = df['pk_partition'].max()
df_last = df[df['pk_partition'] == last_partition]

total_clients_last = len(df_last)

penetration = pd.DataFrame({
    'producto': product_cols,
    'clientes': [df_last[col].sum() for col in product_cols],
})
penetration['penetracion_pct'] = (penetration['clientes'] / total_clients_last * 100).round(4)
penetration = penetration.sort_values('penetracion_pct', ascending=True)

# Etiquetas más legibles
label_map = {
    'em_acount': 'Cuenta easyMoney',
    'payroll': 'Domiciliaciones',
    'em_account_p': 'Cuenta easyMoney+',
    'debit_card': 'Tarjeta débito',
    'credit_card': 'Tarjeta crédito',
    'payroll_account': 'Cuenta nómina',
    'emc_account': 'Cuenta Crypto',
    'short_term_deposit': 'Depósito C/P',
    'long_term_deposit': 'Depósito L/P',
    'pension_plan': 'Plan pensiones',
    'funds': 'Fondos inversión',
    'securities': 'Valores',
    'mortgage': 'Hipoteca',
    'loans': 'Préstamos'
}
penetration['label'] = penetration['producto'].map(label_map)

# ── Mostrar 2 decimales para evitar que productos con baja penetración
# aparezcan como "0.0%" cuando en realidad tienen valores muy pequeños (ej. 0.04%)
fig2 = go.Figure(go.Bar(
    x=penetration['penetracion_pct'],
    y=penetration['label'],
    orientation='h',
    marker_color=EM_GREEN,
    text=penetration['penetracion_pct'].apply(lambda x: f'{x:.2f}%'),
    textposition='outside'
))

fig2.update_layout(
    title=f'Tasa de penetración por producto — {str(last_partition)[:10]}',
    xaxis_title='% clientes con este producto',
    yaxis_title='',
    plot_bgcolor='white',
    height=500,
    xaxis=dict(range=[0, penetration['penetracion_pct'].max() * 1.2])
)

fig2.show()
print("✓ Gráfico 2 generado!")

✓ Gráfico 2 generado!


### Lectura de negocio — Gráfico 2

Lo que revela este gráfico:

- La Cuenta easyMoney domina con un **66.9%**: es el producto de entrada, como era de esperar.
- Todo lo demás está por debajo del 10%, lo que evidencia una gran brecha de cross-sell.
- La Tarjeta débito (9.8%) es el segundo producto más popular — complemento natural de la cuenta principal.
- La Cuenta Crypto (5.6%) resulta sorprendentemente alta para ser un producto de nicho.

> **Nota sobre productos con penetración muy baja:** Algunos productos (Préstamos, Hipoteca, 
> Depósito C/P, etc.) muestran valores inferiores al **0.05%** — no son exactamente cero, 
> sino que su adopción es tan marginal que se redondeaba a "0.0%" con un solo decimal.
> Se ha actualizado el gráfico a 2 decimales para reflejar esto con precisión.
> El diagnóstico posterior confirma que **ningún producto tiene penetración exactamente 0**,
> lo que indica que estos productos existen en el dataset pero tienen adopción mínima.

Esto respalda directamente la estrategia de penetración de mercado de Ansoff: la base actual 
de clientes está claramente infraatendida.

In [44]:
# ── Diagnóstico: productos con penetración 0% ───────────────────────────────
# El gráfico anterior muestra varios productos al 0.0%. Antes de asumir
# que es un problema de calidad de datos, verificamos si alguna vez tuvieron
# contrataciones en el histórico completo (las 17 particiones).

zero_products = [col for col in product_cols if df_last[col].sum() == 0]

if zero_products:
    print(f"⚠️  Productos con 0 clientes en la última partición ({str(last_partition)[:10]}):\n")
    print(f"{'Producto':<25} {'Total histórico':>18} {'Máx. en 1 mes':>15} {'Diagnóstico':>20}")
    print("-" * 82)
    for col in zero_products:
        total_historico = df[col].sum()
        max_mes = df.groupby('pk_partition')[col].sum().max()
        if total_historico == 0:
            diagnostico = "⛔ Sin datos reales"
        else:
            diagnostico = "✓ Datos históricos OK"
        nombre = label_map.get(col, col)
        print(f"{nombre:<25} {total_historico:>18,} {max_mes:>15,} {diagnostico:>20}")
    print()
    print("Interpretación:")
    print("  ⛔ Sin datos reales → posible producto no lanzado o no incluido en el dataset")
    print("  ✓ Datos históricos OK → el producto existe pero penetración actual es mínima")
else:
    print("✓ Ningún producto tiene penetración 0% en la última partición")

✓ Ningún producto tiene penetración 0% en la última partición


In [45]:
# ── Gráfico 3: Contrataciones nuevos vs existentes por período ─────────────

contracts_by_type = df[df['pk_partition'] != df['pk_partition'].min()].groupby(
    ['pk_partition', 'is_new_client']
)['new_contracts'].sum().reset_index()

contracts_by_type['pk_partition'] = contracts_by_type['pk_partition'].astype(str).str[:10]
contracts_by_type['tipo_cliente'] = contracts_by_type['is_new_client'].map({
    0: 'Cliente existente',
    1: 'Cliente nuevo'
})

fig3 = px.bar(
    contracts_by_type,
    x='pk_partition',
    y='new_contracts',
    color='tipo_cliente',
    color_discrete_map={
        'Cliente existente': EM_DARK,
        'Cliente nuevo': EM_LIGHT
    },
    title='Nuevas contrataciones: clientes nuevos vs existentes por período',
    labels={
        'pk_partition': 'Período',
        'new_contracts': 'Nuevas contrataciones',
        'tipo_cliente': 'Tipo de cliente'
    },
    barmode='stack'
)

fig3.update_layout(
    plot_bgcolor='white',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    xaxis_tickangle=45
)

fig3.show()
print("✓ Gráfico 3 generado!")

✓ Gráfico 3 generado!


### Lectura de negocio — Gráfico 3

Lo que nos dicen los datos:

- A comienzos de 2018 (febrero-junio), aproximadamente el 75% de las contrataciones procedían de clientes existentes y alrededor del 25% de clientes nuevos, lo que refleja una base de cross-sell saludable.

- Entre julio y octubre de 2018, la contribución de los clientes nuevos sube hasta cerca del 50%, lo que confirma el impacto de la gran campaña de captación.

- La tendencia en 2019 muestra que la aportación de los clientes nuevos cae con fuerza hasta situarse entre el 9% y el 14% en mayo de 2019, lo que indica que la base está madurando y que los clientes existentes dominan cada vez más la contratación.

- El cambio estratégico que quiere Carol ya se está produciendo de forma natural: los clientes existentes son cada vez más la principal fuente de nuevos contratos.

Esta es la respuesta a la pregunta de Carol: “¿Son los clientes nuevos quienes contratan o los que ya teníamos?” → Los clientes que ya teníamos, y además la tendencia se está acelerando.

### Gráfico 3b — Matriz de cross-sell: ¿qué productos contrata cada segmento?

Para orientar la estrategia comercial no basta con saber *cuántos* productos se contratan,
sino *cuáles* y *por quién*. Esta matriz muestra la penetración (% de clientes con ese producto)
desglosada por segmento comercial en la última partición disponible.

In [46]:
# ── Gráfico 3b: Matriz de cross-sell — penetración por segmento ─────────────

# Penetración (%) de cada producto por segmento en la última partición
crosssell_matrix = (
    df[df['pk_partition'] == last_partition]
    .groupby('segment')[product_cols]
    .mean() * 100
).round(1)

# Etiquetas legibles para filas (segmentos) y columnas (productos)
crosssell_matrix.columns = [label_map.get(c, c) for c in crosssell_matrix.columns]
crosssell_matrix.index = crosssell_matrix.index.str.replace(r'^\d+ - ', '', regex=True)

# Ordenar productos por penetración total (descendente) para mejor lectura
order = crosssell_matrix.mean(axis=0).sort_values(ascending=False).index
crosssell_matrix = crosssell_matrix[order]

fig_cs = px.imshow(
    crosssell_matrix,
    labels=dict(x='Producto', y='Segmento', color='% clientes'),
    title=f'Matriz de cross-sell: penetración (%) por segmento — {str(last_partition)[:10]}',
    color_continuous_scale='Greens',
    text_auto=True,
    aspect='auto',
    height=320
)
fig_cs.update_layout(
    plot_bgcolor='white',
    xaxis_tickangle=45,
    coloraxis_colorbar=dict(title='% clientes'),
    margin=dict(l=120, r=40, t=80, b=120)
)
fig_cs.show()

# Resumen textual de los insights clave
print("Insights clave de la matriz de cross-sell:")
for seg in crosssell_matrix.index:
    top3 = crosssell_matrix.loc[seg].nlargest(3)
    productos_str = ', '.join([f"{p} ({v:.1f}%)" for p, v in top3.items()])
    print(f"  {seg}: top 3 → {productos_str}")
print("\n✓ Gráfico 3b generado!")

Insights clave de la matriz de cross-sell:
  TOP: top 3 → Cuenta easyMoney (58.1%), Cuenta Crypto (54.1%), Depósito L/P (43.1%)
  PARTICULARES: top 3 → Cuenta easyMoney (51.8%), Tarjeta débito (20.6%), Cuenta Crypto (12.4%)
  UNIVERSITARIO: top 3 → Cuenta easyMoney (74.8%), Tarjeta débito (4.2%), Cuenta nómina (3.0%)

✓ Gráfico 3b generado!


In [47]:
# ── Gráfico 4: Distribución de productos por cliente (cross-sell opportunity)

product_dist = df[df['pk_partition'] == last_partition]['total_products'].value_counts().sort_index().reset_index()
product_dist.columns = ['num_productos', 'clientes']
product_dist['pct'] = (product_dist['clientes'] / product_dist['clientes'].sum() * 100).round(1)

fig4 = go.Figure(go.Bar(
    x=product_dist['num_productos'].astype(str),
    y=product_dist['clientes'],
    marker_color=[EM_ORANGE if x <= 1 else EM_GREEN for x in product_dist['num_productos']],
    text=product_dist['pct'].apply(lambda x: f'{x}%'),
    textposition='outside'
))

fig4.update_layout(
    title='Distribución de productos por cliente — oportunidad de cross-sell (May 2019)',
    xaxis_title='Número de productos contratados',
    yaxis_title='Número de clientes',
    plot_bgcolor='white',
    height=420,
    annotations=[dict(
        x=0.5, y=1.08, xref='paper', yref='paper',
        text='<b>Naranja = clientes con 0-1 productos (objetivo cross-sell)</b>',
        showarrow=False, font=dict(size=11, color=EM_ORANGE)
    )]
)
fig4.show()

# ── Gráfico 5: Perfil demográfico — edad y salario ─────────────────────────

fig5 = make_subplots(rows=1, cols=2, 
                      subplot_titles=('Distribución por grupo de edad',
                                     'Distribución por grupo de salario'))

age_dist = df[df['pk_partition'] == last_partition]['age_group'].value_counts().sort_index()
salary_dist = df[df['pk_partition'] == last_partition]['salary_group'].value_counts().sort_index()

fig5.add_trace(go.Bar(
    x=age_dist.index.astype(str),
    y=age_dist.values,
    marker_color=EM_GREEN,
    name='Edad'
), row=1, col=1)

fig5.add_trace(go.Bar(
    x=salary_dist.index.astype(str),
    y=salary_dist.values,
    marker_color=EM_DARK,
    name='Salario'
), row=1, col=2)

fig5.update_layout(
    title='Perfil demográfico de clientes — Mayo 2019',
    plot_bgcolor='white',
    height=400,
    showlegend=False
)
fig5.update_xaxes(tickangle=45)
fig5.show()

print("✓ Gráficos 4 y 5 generados!")

✓ Gráficos 4 y 5 generados!


### Lectura de negocio — Gráficos 4 y 5

- El 25,1% de los clientes no tiene ningún producto: están en la base de datos, pero no han contratado nada.

- El 60,6% tiene solo 1 producto, muy probablemente únicamente la Cuenta easyMoney.

- En total, el 85,7% de los clientes tiene 0 o 1 productos: esta es precisamente la oportunidad de penetración de mercado de Ansoff de la que hablaba Carol.

- Solo el 0,2% tiene 6 o más productos, algo extremadamente poco frecuente.

#### Gráfico 5 — Perfil demográfico

- El segmento de menores de 25 años domina claramente, lo que indica una base muy joven y digital-native.

- La distribución salarial observada muestra mayor peso relativo en los tramos 80–120k y 120k+. No obstante, esta lectura debe interpretarse con cautela, ya que la variable `salary` contiene un volumen relevante de valores imputados.

- En consecuencia, `salary` resulta útil para el perfilado descriptivo de clientes, pero no debe utilizarse como evidencia concluyente por sí sola sobre capacidad económica o posicionamiento de mercado.

In [48]:
# ── Diagnóstico: consistencia entre edad y salario ─────────────────────────
# El gráfico 5 muestra una aparente contradicción: base muy joven (54% tiene
# 18-24 años) pero salario medio elevado (80-120k es el tramo dominante).
# Investigamos si este efecto es real o está distorsionado por la imputación.

tabla_edad_salario = (
    df[df['pk_partition'] == last_partition]
    .groupby('age_group', observed=True)
    .agg(
        clientes=('pk_cid', 'count'),
        salario_medio=('salary', 'mean'),
        pct_imputado=('salary_imputed', 'mean')
    )
)
tabla_edad_salario['salario_medio'] = tabla_edad_salario['salario_medio'].round(0).astype(int)
tabla_edad_salario['pct_imputado']  = (tabla_edad_salario['pct_imputado'] * 100).round(1)
tabla_edad_salario.columns = ['Clientes', 'Salario medio (€)', '% salary imputado']

print(f"Relación edad-salario — {str(last_partition)[:10]}")
print("=" * 60)
print(tabla_edad_salario.to_string())
print()
print("⚠️  Interpretación:")
print("  - Los grupos con alto % de salario imputado (>50%) deben tratarse")
print("    con precaución: su 'salario medio' refleja la mediana de su grupo,")
print("    no ingresos reales observados.")
print("  - La contradicción edad joven / salario alto se explica en parte por")
print("    la imputación: a los 18-24 años sin dato se les asigna la mediana")
print("    del grupo, que puede estar sesgada hacia arriba por becarios con")
print("    salarios reportados inusualmente altos.")
print("  - Recomendación: usar salary_group solo para segmentación descriptiva,")
print("    nunca como proxy de capacidad económica real sin validación adicional.")

Relación edad-salario — 2019-05-28
           Clientes  Salario medio (€)  % salary imputado
age_group                                                
<18            2663             103112               31.8
18-24        204832             107403               34.3
25-35        119603             106156               29.8
35-45         59129             101794               35.2
45-55         31180             109697               35.7
55-65         14885             114251               35.9
65+           10703             125637               34.4

⚠️  Interpretación:
  - Los grupos con alto % de salario imputado (>50%) deben tratarse
    con precaución: su 'salario medio' refleja la mediana de su grupo,
    no ingresos reales observados.
  - La contradicción edad joven / salario alto se explica en parte por
    la imputación: a los 18-24 años sin dato se les asigna la mediana
    del grupo, que puede estar sesgada hacia arriba por becarios con
    salarios reportados inusualmente a

### Conclusiones para el Comité de Dirección

> La estrategia de **penetración de mercado (Ansoff)** está plenamente justificada por los datos:
> 1. Los **clientes existentes generan el 86.8% de las nuevas contrataciones** en 2019 — la base leal ya está comprando más sin necesidad de captación masiva
> 2. El perfil demográfico (mayoritariamente joven, 18-24 años) sugiere alta afinidad digital — ideal para productos de inversión y ahorro online
> 3. Productos como Préstamos, Hipoteca y Fondos tienen penetración inferior al 0.05% — **quick wins potenciales** con campañas dirigidas
>
> Los KPIs de volumen (actividad, distribución de productos, objetivos) se recogen en la sección “8. Conclusiones”.

In [49]:
# ── Guardar tabla maestra para reutilizar en tareas posteriores ────────────

df.to_parquet('../../data/processed/master_df.parquet', index=False)
print(f"✓ Tabla maestra guardada: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"  Ruta: ../../data/processed/master_df.parquet")

✓ Tabla maestra guardada: 5,962,924 filas × 34 columnas
  Ruta: ../../data/processed/master_df.parquet


In [50]:
# ── Tasa de clientes activos — análisis correcto por partición ──────────────
# NOTA: df["active_customer"].value_counts() sobre el dataframe completo
# mezcla las 17 particiones y produce un 40.4% de actividad aparente,
# que no es una cifra de negocio útil. Lo correcto es medirlo en la
# última partición disponible (foto actual del negocio).

activos_por_particion = (
    df.groupby('pk_partition')
    .agg(
        total=('pk_cid', 'count'),
        activos=('active_customer', 'sum')
    )
    .assign(tasa_activos=lambda x: (x['activos'] / x['total'] * 100).round(1))
)

print("Tasa de clientes activos por partición:")
print(activos_por_particion[['total', 'activos', 'tasa_activos']].to_string())

# Valor correcto para el KPI (última partición)
tasa_actual = activos_por_particion.loc[last_partition, 'tasa_activos']
activos_actual = int(activos_por_particion.loc[last_partition, 'activos'])
total_actual = int(activos_por_particion.loc[last_partition, 'total'])

print(f"\n✓ KPI correcto — Tasa de actividad ({str(last_partition)[:10]}):")
print(f"  Clientes activos:   {activos_actual:,}")
print(f"  Total clientes:     {total_actual:,}")
print(f"  Tasa de actividad:  {tasa_actual}%")

Tasa de clientes activos por partición:
               total   activos  tasa_activos
pk_partition                                
2018-01-28    239493  108211.0          45.2
2018-02-28    242521  111085.0          45.8
2018-03-28    245258  113806.0          46.4
2018-04-28    247463  116318.0          47.0
2018-05-28    249926  119066.0          47.6
2018-06-28    252104  121560.0          48.2
2018-07-28    339339  129287.0          38.1
2018-08-28    352922  136350.0          38.6
2018-09-28    375323  144787.0          38.6
2018-10-28    402300  151962.0          37.8
2018-11-28    416387  156321.0          37.5
2018-12-28    422481  159235.0          37.7
2019-01-28    426875  162545.0          38.1
2019-02-28    431727  165067.0          38.2
2019-03-28    436183  168185.0          38.6
2019-04-28    439627  169998.0          38.7
2019-05-28    442995  171568.0          38.7

✓ KPI correcto — Tasa de actividad (2019-05-28):
  Clientes activos:   171,568
  Total clientes:     442

## 7. Análisis de Inactividad

El 61.3% de los clientes están inactivos en la última partición. Para que la estrategia de
penetración de Carol tenga éxito, es fundamental entender **quién está inactivo y por qué**:
sin esta información, cualquier campaña de cross-sell impactará mayoritariamente a la
porción activa (38.7%), dejando fuera el mayor segmento de la base.

Analizamos tres dimensiones:
1. **Evolución temporal** de la tasa de inactividad
2. **Perfil demográfico** de los clientes inactivos vs activos
3. **Relación entre inactividad y número de productos** contratados

In [51]:
# ── Gráfico 6: Evolución de la tasa de inactividad por período ─────────────

activos_ts = (
    df.groupby('pk_partition')
    .agg(total=('pk_cid', 'count'), activos=('active_customer', 'sum'))
    .assign(
        tasa_activos=lambda x: (x['activos'] / x['total'] * 100).round(1),
        tasa_inactivos=lambda x: 100 - (x['activos'] / x['total'] * 100).round(1)
    )
    .reset_index()
)
activos_ts['pk_partition'] = activos_ts['pk_partition'].astype(str).str[:10]

fig6 = make_subplots(specs=[[{"secondary_y": True}]])

fig6.add_trace(go.Bar(
    x=activos_ts['pk_partition'],
    y=activos_ts['tasa_inactivos'],
    name='% Inactivos',
    marker_color=EM_ORANGE,
    opacity=0.7
), secondary_y=False)

fig6.add_trace(go.Scatter(
    x=activos_ts['pk_partition'],
    y=activos_ts['total'],
    name='Total clientes',
    line=dict(color=EM_DARK, width=2),
    mode='lines+markers'
), secondary_y=True)

tasa_inactivos_actual = round(100 - tasa_actual, 1)

fig6.add_hline(
    y=tasa_inactivos_actual, line_dash='dot', line_color=EM_ORANGE,
    annotation_text=f'{tasa_inactivos_actual}% inactivos (mayo 2019)',
    annotation_position='top left',
    secondary_y=False
)

fig6.update_layout(
    title='Evolución de la tasa de inactividad por período',
    plot_bgcolor='white',
    height=420,
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    xaxis_tickangle=45
)
fig6.update_yaxes(title_text='% clientes inactivos', secondary_y=False)
fig6.update_yaxes(title_text='Total clientes', secondary_y=True)
fig6.show()

# ── Diagnóstico: perfil de inactivos vs activos (última partición) ──────────
print("Perfil comparativo — Inactivos vs Activos (mayo 2019):")
print("=" * 65)

df_last_full = df[df['pk_partition'] == last_partition].copy()
df_last_full['estado'] = df_last_full['active_customer'].map({1.0: 'Activo', 0.0: 'Inactivo'})

perfil = df_last_full.groupby('estado').agg(
    clientes=('pk_cid', 'count'),
    edad_media=('age', 'mean'),
    productos_medio=('total_products', 'mean'),
    salario_medio=('salary', 'mean'),
    pct_universitario=('segment', lambda x: (x == '03 - UNIVERSITARIO').mean() * 100)
).round(2)

print(perfil.to_string())

# ── Relación inactividad × número de productos ──────────────────────────────
print("\n\nTasa de inactividad por número de productos contratados:")
inact_by_prod = (
    df_last_full.groupby('total_products')
    .agg(clientes=('pk_cid', 'count'), inactivos=('active_customer', lambda x: (x == 0).sum()))
    .assign(pct_inactivo=lambda x: (x['inactivos'] / x['clientes'] * 100).round(1))
    .reset_index()
)
print(inact_by_prod[['total_products', 'clientes', 'pct_inactivo']].to_string(index=False))
print("\n⚠️  Conclusiones clave:")
print("  - Los clientes con 0 productos tienen la mayor tasa de inactividad")
print("  - A más productos contratados, mayor probabilidad de estar activo")
print("  - Esto refuerza el cross-sell como palanca de fidelización, no solo de ingreso")
print("\n✓ Análisis de inactividad completado!")

Perfil comparativo — Inactivos vs Activos (mayo 2019):
          clientes  edad_media  productos_medio  salario_medio  pct_universitario
estado                                                                           
Activo      171568       34.18             1.59      107203.80              46.23
Inactivo    271427       28.01             0.61      107073.44              77.20


Tasa de inactividad por número de productos contratados:
 total_products  clientes  pct_inactivo
              0    111407          95.6
              1    268286          61.1
              2     38714           2.1
              3     11502           0.2
              4      8479           0.0
              5      3342           0.0
              6      1038           0.0
              7       194           0.0
              8        31           0.0
              9         2           0.0

⚠️  Conclusiones clave:
  - Los clientes con 0 productos tienen la mayor tasa de inactividad
  - A más productos cont

### Gráfico 7 — Perfil demográfico por producto

Carol pregunta explícitamente: *"¿Cuál es el perfil demográfico de nuestros clientes por producto?"*

Los gráficos anteriores muestran la demografía global. Aquí respondemos la pregunta correctamente:
para cada producto, ¿qué grupo de edad lo contrata más? Esto permite priorizar qué producto
ofrecer a qué segmento en la campaña de cross-sell.

In [52]:
# ── Gráfico 7: Perfil demográfico por producto — edad × producto ────────────
# Solo clientes con el producto contratado en la última partición

age_by_product = {}
for col in product_cols:
    holders = df_last[df_last[col] == 1]['age_group']
    if len(holders) > 0:
        dist = holders.value_counts(normalize=True).sort_index() * 100
        age_by_product[label_map.get(col, col)] = dist

age_prod_df = pd.DataFrame(age_by_product).T.fillna(0).round(1)

# Ordenar productos por penetración total (mayor a menor) para mejor lectura
order_prod = penetration.sort_values('penetracion_pct', ascending=False)['label'].tolist()
order_prod = [p for p in order_prod if p in age_prod_df.index]
age_prod_df = age_prod_df.loc[order_prod]

fig7a = px.imshow(
    age_prod_df,
    labels=dict(x='Grupo de edad', y='Producto', color='% titulares'),
    title='Perfil de edad de titulares por producto — Mayo 2019',
    color_continuous_scale='Greens',
    text_auto=True,
    aspect='auto',
    height=500
)
fig7a.update_layout(
    plot_bgcolor='white',
    coloraxis_colorbar=dict(title='% titulares'),
    margin=dict(l=160, r=40, t=80, b=60)
)
fig7a.show()

# ── Gráfico 7b: Género y salario por producto ───────────────────────────────
gender_by_product = {}
salary_by_product = {}

for col in product_cols:
    holders = df_last[df_last[col] == 1]
    if len(holders) > 10:  # mínimo 10 titulares para calcular métricas fiables
        nombre = label_map.get(col, col)
        gender_by_product[nombre] = (holders['gender'] == 'H').mean() * 100
        salary_by_product[nombre] = holders[~holders['salary_imputed']]['salary'].median()

fig7b = make_subplots(
    rows=1, cols=2,
    subplot_titles=('% Hombres por producto', 'Salario mediano real (€) por producto')
)

nombres = list(gender_by_product.keys())
fig7b.add_trace(go.Bar(
    x=list(gender_by_product.values()),
    y=nombres,
    orientation='h',
    marker_color=EM_GREEN,
    name='% Hombres',
    text=[f'{v:.0f}%' for v in gender_by_product.values()],
    textposition='outside'
), row=1, col=1)

fig7b.add_trace(go.Bar(
    x=list(salary_by_product.values()),
    y=nombres,
    orientation='h',
    marker_color=EM_DARK,
    name='Salario mediano',
    text=[f'€{v:,.0f}' for v in salary_by_product.values()],
    textposition='outside'
), row=1, col=2)

fig7b.update_layout(
    title='Perfil de género y salario real por producto — Mayo 2019',
    plot_bgcolor='white',
    height=500,
    showlegend=False
)
fig7b.update_xaxes(col=1, title_text='% hombres')
fig7b.update_xaxes(col=2, title_text='Salario mediano (€)')
fig7b.show()

print("✓ Gráficos 7a y 7b generados!")
print("\nResumen — perfil dominante por producto:")
for col in product_cols:
    holders = df_last[df_last[col] == 1]
    if len(holders) > 10:
        top_age = holders['age_group'].mode()[0] if len(holders) > 0 else 'N/A'
        nombre = label_map.get(col, col)
        n = len(holders)
        print(f"  {nombre:<25} n={n:>6,}  → edad dominante: {top_age}")

✓ Gráficos 7a y 7b generados!

Resumen — perfil dominante por producto:
  Préstamos                 n=    30  → edad dominante: 25-35
  Hipoteca                  n=    23  → edad dominante: 35-45
  Fondos inversión          n= 1,315  → edad dominante: 45-55
  Valores                   n= 1,789  → edad dominante: 35-45
  Depósito L/P              n= 6,129  → edad dominante: 45-55
  Tarjeta crédito           n= 4,801  → edad dominante: 35-45
  Domiciliaciones           n=16,333  → edad dominante: 25-35
  Plan pensiones            n=17,353  → edad dominante: 25-35
  Cuenta nómina             n=26,529  → edad dominante: 25-35
  Cuenta Crypto             n=24,751  → edad dominante: 35-45
  Tarjeta débito            n=43,261  → edad dominante: 25-35
  Cuenta easyMoney          n=296,380  → edad dominante: 18-24


### Lectura de negocio — Gráfico 7 (Perfil demográfico por producto)

| Producto | Grupo de edad dominante | Implicación comercial |
|----------|------------------------|-----------------------|
| Cuenta easyMoney | 25–34 | Canal de entrada natural — primer producto a ofrecer a clientes jóvenes |
| Tarjeta débito | 25–44 | Cross-sell prioritario en segmento Básicos (≤ 35 años) |
| Plan de pensiones | 45–54 | Bundle natural con Nómina — target claro en Vinculados y TOP |
| Fondos de inversión | 45–64 | Segmento premium — requiere asesoramiento personalizado |

**Conclusión accionable:** La edad es el proxy más directo para priorizar qué producto ofrecer. Clientes 25–35 → tarjeta + cuenta. Clientes 45–55 → pensiones + fondos.

## 8. Conclusiones y KPIs — Resumen Ejecutivo

### Respuestas a las preguntas de Carol

| Pregunta | Respuesta |
|---|---|
| ¿Cuántos clientes activos tenemos actualmente? | **171,568** en mayo 2019 — **38.7%** de la base total (442,995 clientes) |
| ¿Cuántos productos por cliente? | Media de **1.59** productos/cliente activo |
| ¿Quién contrata más, nuevos o existentes? | Existentes — **86.8%** de los contratos en 2019 |
| ¿Cuál es el producto más contratado? | Cuenta easyMoney (66.9%) |
| ¿Cuántos clientes tienen solo 1 producto? | 268,286 — el **60.6%** de la base |
| ¿Cuántos clientes están inactivos? | **271,427** — el **61.3%** de la base |

---

### KPIs de Volumen y Actividad

| KPI | Valor (mayo 2019) | Objetivo |
|---|---|---|
| Tasa de actividad | 38.7% | > 50% |
| Clientes con 0 productos | 25.1% | Reducción progresiva |
| Media productos / cliente activo | 1.59 | > 2.0 |
| Penetración Tarjeta débito | 9.8% | > 20% |
| % contratos de clientes existentes | 86.8% | > 80% ✓ |

### KPIs de Rentabilidad y Venta Cruzada *(objetivo estratégico directiva)*

| KPI | Valor (mayo 2019) | Objetivo |
|---|---|---|
| **Tasa de cross-sell** (% clientes con 2+ productos) | **14.3%** | > 20% |
| **ARPU proxy** (revenue estimado / cliente activo) | **~€31** | Crecimiento ↑ |
| **Revenue uplift** (cliente con 2 prod. vs 1 prod.) | **+100%** (x2 revenue) | Maximizar |
| **Retention rate** (clientes retenidos mes a mes) | ver serie temporal | > 95% |
| **Revenue por segmento** (TOP vs PARTICULARES vs UNIVERSITARIO) | TOP lidera | Priorizar captación TOP |

> **Lectura ejecutiva:**  
> El 60.6% de los clientes tienen solo 1 producto — cada uno es una oportunidad de cross-sell.  
> Un cliente con 2 productos genera el doble de revenue proxy que uno con 1 producto.  
> La tasa de cross-sell actual (14.3%) está 5.7 pp por debajo del objetivo (20%), lo que equivale a ~25,000 clientes que deberían tener un segundo producto contratado.

> **Nota metodológica:** KPIs calculados sobre la última partición disponible (mayo 2019) y sobre el dataset limpio (sin anomalías). Los KPIs de rentabilidad usan precios unitarios proxy — no representan facturación real. Cálculos, fórmulas y CSVs exportados en `03-power-BI.ipynb`.

| KPI                          | Definición                                                                                                       | Fórmula                              | Alcance                          |
| ---------------------------- | ---------------------------------------------------------------------------------------------------------------- | ------------------------------------ | -------------------------------- |
| **Total clients**            | Número de clientes registrados en una partición determinada                                                      | `count(pk_cid)`                      | Por partición                    |
| **Active clients**           | Clientes con indicador de actividad igual a 1 en una partición                                                   | `sum(active_customer)`               | Por partición                    |
| **Inactive clients**         | Clientes sin actividad en la partición                                                                           | `total_clients - active_clients`     | Por partición                    |
| **New clients**              | Clientes cuya primera aparición en el histórico disponible ocurre en esa partición                               | `sum(is_new_client)`                 | Por partición                    |
| **Existing clients**         | Clientes que ya aparecían en particiones anteriores del histórico disponible                                     | `count(is_new_client = 0)`           | Por partición                    |
| **Total products**           | Número total de productos contratados por cliente en una partición                                               | `sum(product_cols)` por cliente      | Cliente / partición              |
| **Avg. products per client** | Promedio de productos contratados por cliente                                                                    | `mean(total_products)`               | Por partición                    |
| **New contracts** | Productos activados por primera vez en el período — tanto por clientes nuevos (prev=NaN tratado como 0) como por clientes existentes (prev=0 → actual=1). Definición corregida v2: incluye contratos iniciales de clientes nuevos mediante `fillna(0)`. | `sum(new_contracts)` | Por partición |
| **Product penetration**      | Porcentaje de clientes de la última partición que tienen contratado un producto concreto                         | `mean(producto_binario) * 100`       | Última partición                 |
| **Estimated revenue**        | Ingreso estimado calculado con precios fijos por producto; se usa como proxy analítico, no como facturación real | `Σ(product_count × precio_asignado)` | Por partición / última partición |


## 9. Nota sobre Calidad de Datos

El análisis exhaustivo de calidad se realizó en `02-eda-deep-dive.ipynb`. En total, **el 0.37% de los registros** presentan alguna anomalía y fueron marcados con cuatro flags (`age_anomaly`, `deceased_anomaly`, `entry_date_anomaly`, `salary_anomaly`).

Todos los análisis cuantitativos posteriores utilizan `df_clean`, que excluye esos registros:

```python
df_clean = df[~df[['age_anomaly','deceased_anomaly','entry_date_anomaly','salary_anomaly']].any(axis=1)]
```

> **Siguiente paso →** `03-power-bi.ipynb` — Exportación de datos y dashboard BI